# 01 — UC2 Centralized baseline on the new (label-skew) partitions

Re-evaluates the Centralized upper-bound on the **new** partitions so it can be
compared, on the **same test set**, against FedAvg (`02b`) and FedGen.

Why this is needed: your original `results/centralized/...` were trained and
tested on the **old AP-ID** partitions (`UC2Utils.make_args` hardcodes that
root). The new partitions live elsewhere, so the old Centralized is the wrong
baseline for the new experiments — not invalid, just measured against a
different yardstick.

Why it's cheap: the `Centralized` server pools **all** clients' training data
into one set, and that pool is ~the same ~88.9k windows at every α. So the
centralized *model* is essentially α-invariant; per-α differences come only from
the **test split**. This notebook is really a re-evaluation, not a real retrain.

Results are written to `results/newpart/centralized/...` (client_local) or
`results/newpart_global/centralized/...` (global) — exactly where `02b`'s
gap-to-Centralized cell looks for them. Your original AP-ID results are untouched.


In [ ]:
import sys, os
sys.path.append("..")
import UC2Utils as uc2
sys.path.insert(0, uc2.LIB_DIR)

import numpy as np
import matplotlib.pyplot as plt
import pickle, json
import torch

print(f"Device: {uc2.DEFAULT_CONFIG['device']}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0),
          f"| free {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")

## Configuration

In [ ]:
ALPHAS = [0.01, 0.1, 0.5] # Since centralized is alpha-invariant, we can just run it accros 3 alphas for the sake of comparison with the other algorithms. (We could also just run it once at a single alpha, but this way we get to see the flat line in the results.)
MODEL  = "lstm"

# Must MATCH the variant you ran (and will run) in 02b / FedGen:
#   "client_local" -> new_partitions/        -> results/newpart/
#   "global"       -> new_partitions_global/ -> results/newpart_global/
PARTITION_VARIANT = "client_local"

# First run on a given variant: True so it TRAINS instead of loading stale pkls.
RETRAIN = False

# Centralized config. Mirrors notebook 01 (single pooled model; α only changes
# the test split). early_stopping_patience kept high like the original.
OVERRIDES = dict(
    num_glob_iters=200,
    local_epochs=20,
    num_users=20,
    early_stopping_patience=60,
    batch_size=32,
)

## Partition wiring

Identical mechanism to `02b`: redirect `dataset_path` to the chosen variant and
route `result_path` to a per-variant results folder. `run_experiment` is
otherwise unchanged.

In [ ]:
# config: folder names must match what notebook 00 wrote
VARIANT_SUBDIR  = {"client_local": "new_partitions",
                   "global":       "new_partitions_global"}
VARIANT_RESULTS = {"client_local": "newpart",
                   "global":       "newpart_global"}
assert PARTITION_VARIANT in VARIANT_SUBDIR, PARTITION_VARIANT

_NP_LOOKBACK = uc2.DEFAULT_CONFIG["lookback"]   # 60
_NP_STEPS    = uc2.DEFAULT_CONFIG["steps"]      # 1
_NP_SUBDIR   = VARIANT_SUBDIR[PARTITION_VARIANT]
_NP_RESULTS  = VARIANT_RESULTS[PARTITION_VARIANT]

_orig_make_args     = uc2.make_args
_orig_result_exists = uc2.result_exists
_orig_load_result   = uc2.load_result

_NEWPART_ON = {"flag": False}

def _newpart_dataset_path():
    return os.path.join(uc2.DATA_PART, _NP_SUBDIR,
                        f"lookback_{_NP_LOOKBACK}", f"steps_{_NP_STEPS}")

def _redirect_result_path(result_path, algorithm, alpha, model):
    base = os.path.join(uc2.RESULTS, _NP_RESULTS)
    if result_path is None:
        return os.path.join(base, algorithm.lower(), f"alpha_{alpha}", model, "rep_0")
    rel = os.path.relpath(os.path.abspath(result_path), os.path.abspath(uc2.RESULTS))
    return os.path.join(base, rel)

def _patched_make_args(algorithm, alpha, result_path=None, **overrides):
    args = _orig_make_args(algorithm, alpha, result_path=result_path, **overrides)
    if _NEWPART_ON["flag"]:
        args.dataset_path = _newpart_dataset_path()
        model = {**uc2.DEFAULT_CONFIG, **overrides}["model"]
        new_rp = _redirect_result_path(result_path, algorithm, alpha, model)
        new_rp = os.path.relpath(new_rp)          # lib dislikes spaces in abs paths
        os.makedirs(new_rp, exist_ok=True)
        args.result_path = new_rp
    return args

def _patched_result_exists(algorithm, alpha, model="lstm"):
    if _NEWPART_ON["flag"]:
        p = os.path.join(uc2.RESULTS, _NP_RESULTS, algorithm.lower(),
                         f"alpha_{alpha}", model, "rep_0", "full_results.pkl")
        return os.path.exists(p)
    return _orig_result_exists(algorithm, alpha, model)

def _patched_load_result(algorithm, alpha, model="lstm"):
    if _NEWPART_ON["flag"]:
        p = os.path.join(uc2.RESULTS, _NP_RESULTS, algorithm.lower(),
                         f"alpha_{alpha}", model, "rep_0", "full_results.pkl")
        with open(p, "rb") as f:
            return pickle.load(f)
    return _orig_load_result(algorithm, alpha, model)

def use_new_partitions(on=True):
    """Toggle the selected new-partition variant for subsequent run_experiment calls."""
    _NEWPART_ON["flag"] = bool(on)
    uc2.make_args     = _patched_make_args     if on else _orig_make_args
    uc2.result_exists = _patched_result_exists if on else _orig_result_exists
    uc2.load_result   = _patched_load_result   if on else _orig_load_result
    state = f"NEW [{PARTITION_VARIANT}]" if on else "ORIGINAL AP-ID"
    print(f"[wiring] partitions = {state}")
    if on:
        dp = _newpart_dataset_path()
        print(f"[wiring] dataset_path -> {dp}")
        print(f"[wiring] results      -> {os.path.join(uc2.RESULTS, _NP_RESULTS, '...')}")
        miss = [a for a in ALPHAS if not os.path.exists(
            os.path.join(dp, f"u{uc2.DEFAULT_CONFIG['n_users']}-alpha{a}-ratio1",
                         "train", "train.pt"))]
        if miss:
            print(f"[wiring] !! MISSING partitions for alpha={miss} -- "
                  f"run notebook 00 generation for variant '{PARTITION_VARIANT}' first!")

use_new_partitions(True)

## VRAM check (optional)

In [ ]:
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")
    print(f"Free  VRAM: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")
else:
    print("CUDA not available — running on CPU.")

## Train / evaluate Centralized for each α

Same structure as notebook 01 and `02b`. On the first run of a variant keep
`RETRAIN=True`. The Centralized model is α-invariant by construction, so the
training curves will look near-identical across α; what differs is the test
split (and therefore the reported MAE).

In [ ]:
results_centralized = {}

for alpha in ALPHAS:
    print(f"\n{'='*60}\n  CENTRALIZED [{PARTITION_VARIANT}] — α={alpha}\n{'='*60}")
    if not RETRAIN and uc2.result_exists("Centralized", alpha):
        print("  [done] loading cached result.")
        results_centralized[alpha] = uc2.load_result("Centralized", alpha)
        continue
    try:
        server, result = uc2.run_experiment(algorithm="Centralized", alpha=alpha, **OVERRIDES)
        results_centralized[alpha] = result
        glob = result["metrics"].get("glob_test_metric", [])
        if glob:
            bi = int(np.argmin([m.get("mae", float("inf")) for m in glob]))
            b = glob[bi]
            print(f"  best round {bi}: MAE(scaled)={b.get('mae', float('nan')):.4f} "
                  f"MAE(scaled)={b.get('mae', float('nan')):.4f}")
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f"  [ERROR] {e}")

## (Optional) Fast path — train once, re-evaluate across α

Because the pooled training set is α-invariant, you can train Centralized at a
**single** α and then re-evaluate that one trained model on every other α's test
split, instead of retraining six times. This is OFF by default (the loop above
already produced per-α results). Turn it on only if you are time-constrained and
want to skip redundant training.

It works by training once with `run_experiment`, then for each remaining α
swapping the server's per-user test loaders for that α's test data and calling
`server.test()` again. If anything in your library version makes this brittle,
just use the standard loop above — the results are equivalent.

In [ ]:
USE_FAST_PATH = False   # set True to train once and re-evaluate across alphas

if USE_FAST_PATH:
    from utils.model_utils import read_data, read_user_data
    from torch.utils.data import DataLoader

    base_alpha = ALPHAS[0]   # a mid alpha; choice doesn't matter (alpha-invariant)
    print(f"[fast] training Centralized ONCE at α={base_alpha} ...")
    server, base_result = uc2.run_experiment(algorithm="Centralized", alpha=base_alpha, **OVERRIDES)
    results_centralized[base_alpha] = base_result

    def _reeval_on_alpha(server, alpha):
        """Swap the single Centralized user's test set for `alpha`'s test split and re-test."""
        args = uc2.make_args("Centralized", alpha, **OVERRIDES)   # patched -> new variant path
        data = read_data(args.dataset, args.dataset_path)
        total_users = len(data[0])
        agg_test = []
        for i in range(total_users):
            _id, _train, _test = read_user_data(i, data, dataset=args.dataset, model=args.model)
            agg_test.extend(_test)
        u = server.users[0]
        u.test_data = agg_test
        u.test_samples = len(agg_test)
        u.testloader = DataLoader(agg_test, u.batch_size * 100, drop_last=False)
        u.testloaderfull = DataLoader(agg_test, u.test_samples)
        # rebuild the dataset_path_prefix used by inverse-scaling so it points at this alpha
        ds_ = args.dataset.lower().replace("user","").replace("alpha","").replace("ratio","").split("-")
        user_, alpha_, ratio_ = ds_[1], ds_[2], ds_[3]
        u.dataset_path_prefix = os.path.join(args.dataset_path,
                                             f"u{user_}-alpha{alpha_}-ratio{ratio_}")
        per_user = uc2.evaluate_server(server)
        # store in the same result schema as run_experiment, then persist
        result = {"algorithm": "Centralized", "alpha": alpha, "model": args.model,
                  "config": base_result["config"], "metrics": base_result["metrics"],
                  "per_user": per_user, "n_rounds": base_result["n_rounds"]}
        rp = os.path.join(uc2.RESULTS, _NP_RESULTS, "centralized",
                          f"alpha_{alpha}", MODEL, "rep_0")
        os.makedirs(rp, exist_ok=True)
        with open(os.path.join(rp, "full_results.pkl"), "wb") as f:
            pickle.dump(result, f)
        return result

    for alpha in ALPHAS:
        if alpha == base_alpha:
            continue
        print(f"[fast] re-evaluating trained model on α={alpha} test split ...")
        try:
            results_centralized[alpha] = _reeval_on_alpha(server, alpha)
        except Exception as e:
            import traceback; traceback.print_exc()
            print(f"  [fast ERROR] {e} — fall back to the standard loop for this α.")
else:
    print("[fast] disabled — using per-α results from the standard loop above.")

## Training convergence (sanity)

In [ ]:
uc2.setup_plot_style()
fig, ax = plt.subplots(figsize=(10, 5))
for alpha, r in sorted(results_centralized.items()):
    maes = [m.get("mae") for m in r["metrics"].get("glob_test_metric", [])]
    ax.plot(maes, label=f"α={alpha}", lw=2)
ax.set_ylim(0, .6)
ax.set_xlabel("Epoch"); ax.set_ylabel("MAE (MB)")
ax.set_title(f"Centralized convergence — {PARTITION_VARIANT}")
ax.legend(); plt.tight_layout(); plt.show()

## Baseline summary

Reports both scaled and unscaled best MAE. **Read scaled MAE as primary** until
the inverse-scaling in `userbase.test()` is verified — on this dataset the
unscaled metric decouples from the scaled one at low α (frozen `unscaled_mae`
while `mae` improves), so unscaled MB values are provisional.

In [ ]:
def best_metrics(r):
    glob = r["metrics"].get("glob_test_metric", [])
    if not glob:
        return float("nan"), float("nan")
    bi = int(np.argmin([m.get("mae", float("inf")) for m in glob]))
    return glob[bi].get("mae", float("nan")), glob[bi].get("mae", float("nan"))

print(f"{'α':<8}{'best MAE (scaled)':>20}{'best MAE (unscaled)':>22}{'rounds':>8}")
for a in sorted(results_centralized):
    r = results_centralized[a]
    sc, un = best_metrics(r)
    print(f"{a:<8}{sc:>20.4f}{un:>22.4f}{r.get('n_rounds', 0):>8}")

print("\nNote: Centralized is alpha-invariant by construction; small differences")
print("across alpha reflect the differing TEST split, not a different model.")

## Done

Centralized baseline saved under `results/{newpart|newpart_global}/centralized/...`.

Next:
1. Make sure `02b` used the **same** `PARTITION_VARIANT`.
2. Re-run `02b`'s **gap-to-Centralized** cell — it will now find these results.
3. For the other variant, set `PARTITION_VARIANT="global"` here and in `02b`, and
   re-run both.
4. Then proceed to FedGen on the same variant.


In [ ]:
# C1 verification: does scaled-best and unscaled-best pick the SAME round?
for a, r in sorted(results_centralized.items()):
    glob = r["metrics"].get("glob_test_metric", [])
    if not glob: continue
    i_scaled   = int(np.argmin([m.get("mae", float("inf"))          for m in glob]))
    i_unscaled = int(np.argmin([m.get("unscaled_mae", float("inf")) for m in glob]))
    flag = "" if i_scaled == i_unscaled else "  <-- DIFFERENT round picked!"
    print(f"α={a}: scaled-best round={i_scaled}, unscaled-best round={i_unscaled}{flag}")